In [ ]:
import os
import sys
import asyncio
import smtplib
from datetime import datetime
from typing import List, Literal
from pydantic import BaseModel, Field
from email.mime.text import MIMEText

from langchain.agents import Tool, initialize_agent, AgentType
from langchain_openai import AzureChatOpenAI
from langchain.callbacks.base import BaseCallbackHandler
from langchain.memory import ConversationBufferMemory

# === NEW: Project Management SDKs ===
from asana import Client as AsanaClient
from atlassian import Jira
import requests  # for ClickUp

# =============================
# Azure LLM Configuration
# =============================

os.environ["AZURE_OPENAI_API_KEY"] = "<OPEN_API_KEY>"
os.environ["AZURE_OPENAI_ENDPOINT"] = "<AZURE_OPENAI_ENDPOINT>"
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "<AZURE_OPENAI_DEPLOYMENT_NAME>"
os.environ["OPENAI_API_VERSION"] = "<OPENAI_API_VERSION>"

# =============================
# Credentials (env vars)
# =============================
ASANA_TOKEN = os.getenv("ASANA_TOKEN")
JIRA_URL = os.getenv("JIRA_URL")
JIRA_USER = os.getenv("JIRA_USER")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
CLICKUP_TOKEN = os.getenv("CLICKUP_TOKEN")

# =============================
# LLM State Tracking
# =============================

class LLMState:
    def __init__(self):
        self.state = "Listening"
        self.last_update = datetime.now()

    def update(self, new_state: str):
        self.state = new_state
        self.last_update = datetime.now()

    def get_state(self):
        return {"state": self.state, "last_update": self.last_update}

class RealTimeLLMCallback(BaseCallbackHandler):
    def __init__(self, llm_state: LLMState):
        self.llm_state = llm_state
        self.last_state = None

    def _print_state(self, state: str):
        if state != self.last_state:
            if state == "active":
                print("[STATUS] Working…")
            elif state == "Listening":
                print("[STATUS] Listening ✅")
            elif state == "error":
                print("[STATUS] Error ❌")
            self.last_state = state

    async def on_llm_start(self, *args, **kwargs):
        self.llm_state.update("active")
        self._print_state("active")

    async def on_llm_end(self, *args, **kwargs):
        self.llm_state.update("Listening")
        self._print_state("Listening")

    async def on_llm_error(self, error: Exception, *args, **kwargs):
        self.llm_state.update("error")
        self._print_state("error")

# =============================
# Conversation Models
# =============================

class ConversationEntry(BaseModel):
    speaker: Literal["Human", "AI"]
    message: str
    timestamp: datetime = Field(default_factory=datetime.now)
    context_type: Literal["AI_Context", "AI_Followup", "Human_Only"]

class ConversationHistory(BaseModel):
    entries: List[ConversationEntry] = []

    def add(self, speaker: str, message: str, context_type: str):
        self.entries.append(ConversationEntry(
            speaker=speaker, message=message, context_type=context_type
        ))

    def show(self):
        for e in self.entries:
            print(f"[{e.timestamp:%H:%M:%S}] {e.speaker} ({e.context_type}): {e.message}")

    def recall_last_question(self) -> str:
        human_msgs = [e for e in self.entries if e.speaker == "Human" and e.context_type != "Human_Only"]
        if not human_msgs:
            return "No previous question found."
        return human_msgs[-1].message

# =============================
# Tools (Extended with PM APIs)
# =============================

def make_tools(agent_name: str, history: ConversationHistory):
    tools = [
        Tool(
            name="Echo",
            func=lambda x: f"{agent_name} received: {x}",
            description="Echoes back the user's input"
        ),
        Tool(
            name="RecallLastQuestion",
            func=lambda _: f'The last question you asked was: "{history.recall_last_question()}"',
            description="Retrieves the last human question."
        )
    ]

    # --- Asana ---
    if ASANA_TOKEN:
        asana_client = AsanaClient.access_token(ASANA_TOKEN)

        def get_asana_tasks(workspace_gid: str, project_gid: str):
            tasks = asana_client.tasks.find_by_project(project_gid, {"workspace": workspace_gid})
            return [t["name"] for t in tasks]

        tools.append(Tool(
            name="AsanaTasks",
            func=lambda _: str(get_asana_tasks("<WORKSPACE_ID>", "<PROJECT_ID>")),
            description="Fetches tasks from Asana project"
        ))

    # --- Jira ---
    if JIRA_URL and JIRA_USER and JIRA_API_TOKEN:
        jira = Jira(
            url=JIRA_URL,
            username=JIRA_USER,
            password=JIRA_API_TOKEN
        )

        def get_jira_issues(jql="project=TEST ORDER BY created DESC"):
            issues = jira.jql(jql).get("issues", [])
            return [i["fields"]["summary"] for i in issues]

        tools.append(Tool(
            name="JiraIssues",
            func=lambda _: str(get_jira_issues()),
            description="Fetches issues from Jira project"
        ))

    # --- ClickUp ---
    if CLICKUP_TOKEN:
        def get_clickup_tasks(list_id: str):
            url = f"https://api.clickup.com/api/v2/list/{list_id}/task"
            headers = {"Authorization": CLICKUP_TOKEN}
            r = requests.get(url, headers=headers)
            return [t["name"] for t in r.json().get("tasks", [])]

        tools.append(Tool(
            name="ClickUpTasks",
            func=lambda _: str(get_clickup_tasks("<LIST_ID>")),
            description="Fetches tasks from ClickUp list"
        ))

    return tools

# =============================
# Notification Manager
# =============================

class Notifier:
    def __init__(self, email_sender=None, email_password=None, email_receiver=None):
        self.email_sender = email_sender
        self.email_password = email_password
        self.email_receiver = email_receiver

    def notify_console(self, message: str):
        print(f"[NOTIFICATION] {message}")

    def notify_email(self, subject: str, body: str):
        if not all([self.email_sender, self.email_password, self.email_receiver]):
            self.notify_console(f"(Email skipped) {subject}: {body}")
            return
        try:
            msg = MIMEText(body)
            msg["Subject"] = subject
            msg["From"] = self.email_sender
            msg["To"] = self.email_receiver

            with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
                server.login(self.email_sender, self.email_password)
                server.sendmail(self.email_sender, self.email_receiver, msg.as_string())
            self.notify_console("Email notification sent successfully.")
        except Exception as e:
            self.notify_console(f"Email failed: {e}")

# =============================
# Async Input Helper
# =============================

async def ainput(prompt: str = "", timeout: int = None) -> str:
    try:
        return await asyncio.wait_for(asyncio.to_thread(input, prompt), timeout=timeout)
    except asyncio.TimeoutError:
        return None

# =============================
# Agent Creation
# =============================

def create_agent(agent_name: str, llm, history: ConversationHistory, with_prompt=False, with_memory=False):
    tools = make_tools(agent_name, history)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True) if with_memory else None

    agent = initialize_agent(
        tools,
        llm,
        agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
        memory=memory,
        verbose=True,
        handle_parsing_errors=True
    )
    print(f"[INFO] New agent '{agent_name}' created! (Prompt={with_prompt}, Memory={with_memory})")
    return agent

# =============================
# MetaAgent Manager
# =============================

class MetaAgent:
    def __init__(self, llm, notifier: Notifier):
        self.agents = {}
        self.llm = llm
        self.history = ConversationHistory()
        self.current_agent = None
        self.notifier = notifier

    def handle_request(self, request: str):
        parts = request.strip().split()
        if not parts:
            return "No command provided."

        cmd = parts[0].lower()

        if cmd == "create":
            if len(parts) < 3 or parts[1].lower() != "agent":
                return "Usage: create agent <name> [with prompt] [with memory]"
            raw_name = parts[2]
            agent_name = raw_name.strip("<>[]").capitalize()
            rest = " ".join(parts[3:]).lower()
            with_prompt = "with prompt" in rest
            with_memory = "with memory" in rest
            if agent_name in self.agents:
                return f"Agent '{agent_name}' already exists."
            self.agents[agent_name] = create_agent(agent_name, self.llm, self.history, with_prompt, with_memory)
            self.current_agent = agent_name
            return f"Agent '{agent_name}' created with Prompt={with_prompt}, Memory={with_memory}."

        elif cmd == "delete":
            if len(parts) < 3 or parts[1].lower() != "agent":
                return "Usage: delete agent <name>"
            agent_name = parts[2].strip("<>[]").capitalize()
            if agent_name not in self.agents:
                return f"Agent '{agent_name}' does not exist."
            del self.agents[agent_name]
            return f"Agent '{agent_name}' deleted."

        elif cmd == "list":
            return f"Current Agents: {list(self.agents.keys())}"

        elif cmd == "history":
            self.history.show()
            return "=== End of History ==="

        elif cmd == "use":
            if ":" in request:
                name_part, msg_part = request.split(":", 1)
                agent_name = name_part.replace("use", "").strip("<>[] ").capitalize()
                message = msg_part.strip()
                if agent_name not in self.agents:
                    return f"Agent '{agent_name}' does not exist."
                self.current_agent = agent_name
                return agent_name, message
            else:
                return "Usage: use <agent_name>: <message>"

        elif self.current_agent:  # if agent is set, treat input as message
            return self.current_agent, request

        else:
            return f"Unknown command '{cmd}'. Try again."

# =============================
# Main Interactive Loop
# =============================

async def main():
    print("=== MetaAgent Control ===")
    print("Commands:")
    print("  create agent <name> [with prompt] [with memory]")
    print("  delete agent <name>")
    print("  use <name>: <message>")
    print("  list agents")
    print("  history")
    print("  quit")

    llm_state = LLMState()
    callback = RealTimeLLMCallback(llm_state)

    llm = AzureChatOpenAI(
        deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        streaming=True,
        callbacks=[callback],
        temperature=0.7
    )

    notifier = Notifier(
        email_sender=os.getenv("EMAIL_SENDER"),
        email_password=os.getenv("EMAIL_PASSWORD"),
        email_receiver=os.getenv("EMAIL_RECEIVER")
    )

    meta_agent = MetaAgent(llm, notifier)

    while True:
        command = (await ainput("\n>>> ")).strip()
        if command.lower() == "quit":
            print("Exiting MetaAgent.")
            break

        result = meta_agent.handle_request(command)

        if isinstance(result, tuple):
            agent_name, message = result

            print(f"Pending approval for: {message}")
            approval = await ainput(
                f"Send to {agent_name}? (y=send / n=skip / r=rephrase / c=cancel / h=human-only): ",
                timeout=60
            )

            if approval is None:
                meta_agent.notifier.notify_console("User unavailable for 60s. Auto-responding on their behalf.")
                meta_agent.notifier.notify_email(
                    subject=f"MetaAgent Auto-Reply from {agent_name}",
                    body=f"User was unavailable. The agent auto-replied to: '{message}'"
                )
                continue

            if approval == "n":
                print("Skipped by human.")
                continue
            elif approval == "c":
                print("Cancelled.")
                continue
            elif approval == "r":
                message = (await ainput("Enter rephrased message: ")).strip()
            elif approval == "h":
                meta_agent.history.add("Human", message, "Human_Only")
                print("[HUMAN-ONLY LOGGED]")
                continue

            try:
                meta_agent.history.add("Human", message, "AI_Context")
                response = await meta_agent.agents[agent_name].ainvoke(message)
                output = response["output"]
                print(f"{agent_name} Response: {output}")
                meta_agent.history.add("AI", output, "AI_Followup")

            except Exception as e:
                llm_state.update("error")
                print("LLM Error:", e)

            print("State snapshot:", llm_state.get_state())
        else:
            print(result)

# =============================
# Run
# =============================

if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.get_event_loop().run_until_complete(main())
    else:
        asyncio.run(main())


=== MetaAgent Control ===
Commands:
  create agent <name> [with prompt] [with memory]
  delete agent <name>
  use <name>: <message>
  list agents
  quit


/var/folders/k4/gtw_xk_964b4l72k2ynn93b80000gn/T/ipykernel_40018/3507690413.py:177: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import AzureChatOpenAI``.
  llm = AzureChatOpenAI(



>>> create agent David
[INFO] New agent 'David' created! (Prompt=False, Memory=False)
Agent 'David' created with Prompt=False, Memory=False.


/var/folders/k4/gtw_xk_964b4l72k2ynn93b80000gn/T/ipykernel_40018/3507690413.py:92: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(
